# Gen3 Embeddings Demo

> This will demonstrate how to create and retrieve embeddings in bulk from Gen3

First, let's install the Gen3 Python Software Development Kit (SDK), which includes a command line interface (CLI).

In [41]:
%pip install --upgrade pip
%pip install gen3 --upgrade

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [67]:
# we also need pandas for some nice visualizations of output files
import pandas as pd

If you are running the Gen3 AI support (specifically the Gen3 Embeddings API) locally, you can set some variables to point to the right credentials file and the service.

If you are trying to interact with a production instance, just leave the default `ai`. e.g. don't uncomment - the default credentials setup should point you to the right place.

In [ ]:
ai = "ai"
url_prefix = ""

# TODO: Comment these out
# ai = "--auth ~/.gen3/local_helm_test_user.json --endpoint 'http://127.0.0.1:4142' ai --api-prefix ''"
# url_prefix = "http://127.0.0.1:4142"

## Gen3 AI Embeddings CLI

In [44]:
!gen3 ai embeddings --help

Usage: gen3 ai embeddings [OPTIONS] COMMAND [ARGS]...

  For working with embeddings

Options:
  --help  Show this message and exit.

Commands:
  collections  For working with embeddings collections
  convert      Convert a Gen3 Embeddings Manifest into a Gen3 Indexing...
  delete       [Not Implemented Yet] Deletes specified embeddings data in...
  embed-files  Chunk files and create embeddings locally, then submit to...
  publish      Publish a manifest of already-embedded objects to a Gen3...
  read         [Not Implemented Yet] Reads embeddings data from Gen3...


In [45]:
!gen3 ai embeddings collections --help

Usage: gen3 ai embeddings collections [OPTIONS] COMMAND [ARGS]...

  For working with embeddings collections

Options:
  --help  Show this message and exit.

Commands:
  create  Create a new embeddings collection.
  delete  Delete a embeddings collection.
  read    Read embeddings collections.


Get a response from the service by reading collections. 

> IMPORTANT: You need appropriate permissions to read (and for future sections: write).
> So these commands may be empty or fail unless you have those authorizations in the environment your credentials are for.

In [46]:
!gen3 $ai embeddings collections read

Listing all embeddings collections...
Found 11 collection(s)
- id: 1
  collection_name: noaccess
  description: noaccess collection
  dimensions: 3
  vector_type: vector
  created_at: 2026-05-27T19:53:09.635886Z
  updated_at: 2026-05-27T19:53:09.635886Z
  self: /vectorstore/collections/noaccess
- id: 2
  collection_name: public
  description: public collection
  dimensions: 3
  vector_type: vector
  created_at: 2026-05-27T19:53:09.635886Z
  updated_at: 2026-05-27T19:53:09.635886Z
  self: /vectorstore/collections/public
- id: 3
  collection_name: internal
  description: internal collection
  dimensions: 3
  vector_type: vector
  created_at: 2026-05-27T19:53:09.635886Z
  updated_at: 2026-05-27T19:53:09.635886Z
  self: /vectorstore/collections/internal
- id: 4
  collection_name: d3vector
  description: d3 vector collection
  dimensions: 3
  vector_type: vector
  created_at: 2026-05-27T19:53:09.635886Z
  updated_at: 2026-05-27T19:53:09.635886Z
  self: /vectorstore/collections/d3vector
- id

## Creating Embeddings Collections

This will show creation of collections and embeddings. So you need write permission or this will fail. You can run the Gen3 Embeddings API locally to test this out. See the [Gen3 AI repo README](https://github.com/uc-cdis/gen3-ai) for more information.

In [47]:
!gen3 $ai embeddings collections delete "ctds-github-md"
!gen3 $ai embeddings collections create "ctds-github-md" --dimensions 384 --description "All markdown from CTDS Github"

Deleting collection 'ctds-github-md'...
Successfully deleted collection 'ctds-github-md'!
Creating collection 'ctds-github-md' with 384 dimensions...
Collection created successfully!
{
  "id": 140,
  "collection_name": "ctds-github-md",
  "description": "All markdown from CTDS Github",
  "dimensions": 384,
  "vector_type": "vector",
  "created_at": "2026-06-01T21:44:28.396532Z",
  "updated_at": "2026-06-01T21:44:28.396532Z",
  "self": "/vectorstore/collections/ctds-github-md"
}


In [48]:
!gen3 $ai embeddings collections read "ctds-github-md" 

Reading embeddings collection ctds-github-md...
Found ctds-github-md collection
- id: 140
  collection_name: ctds-github-md
  description: All markdown from CTDS Github
  dimensions: 384
  vector_type: vector
  created_at: 2026-06-01T21:44:28.396532Z
  updated_at: 2026-06-01T21:44:28.396532Z
  self: /vectorstore/collections/ctds-github-md


In [49]:
# try to delete collections that might already exist
!gen3 $ai embeddings collections delete "expr"
!gen3 $ai embeddings collections delete "hist"
!gen3 $ai embeddings collections delete "summ"

Deleting collection 'expr'...
Successfully deleted collection 'expr'!
Deleting collection 'hist'...
Successfully deleted collection 'hist'!
Deleting collection 'summ'...
Successfully deleted collection 'summ'!


Let's create some more example collections.

> IMPORTANT: You need permission to create and manage these collections *before* running the commands. So ensure the Gen3 operator adds these resources to the `user.yaml` and provides your user permission to them. If you are running Gen3 yourself, you can see the [Gen3 AI repo README](https://github.com/uc-cdis/gen3-ai) for more information on how to set up appropriate auth.

In [50]:
!gen3 $ai embeddings collections create "expr" --dimensions 256 --description "test expr data"
!gen3 $ai embeddings collections create "hist" --dimensions 1536 --description "test hist data"

Creating collection 'expr' with 256 dimensions...
Collection created successfully!
{
  "id": 141,
  "collection_name": "expr",
  "description": "test expr data",
  "dimensions": 256,
  "vector_type": "vector",
  "created_at": "2026-06-01T21:44:32.235173Z",
  "updated_at": "2026-06-01T21:44:32.235173Z",
  "self": "/vectorstore/collections/expr"
}
Creating collection 'hist' with 1536 dimensions...
Collection created successfully!
{
  "id": 142,
  "collection_name": "hist",
  "description": "test hist data",
  "dimensions": 1536,
  "vector_type": "vector",
  "created_at": "2026-06-01T21:44:32.940199Z",
  "updated_at": "2026-06-01T21:44:32.940199Z",
  "self": "/vectorstore/collections/hist"
}


You can also create collections of larger dimensional size. This will use a `vector_type` of `halfvec` to fit it into the underlying database.

In [51]:
!gen3 $ai embeddings collections create "summ" --dimensions 4096 --description "test summ data"

Creating collection 'summ' with 4096 dimensions...
Collection created successfully!
{
  "id": 143,
  "collection_name": "summ",
  "description": "test summ data",
  "dimensions": 4096,
  "vector_type": "halfvec",
  "created_at": "2026-06-01T21:44:33.648493Z",
  "updated_at": "2026-06-01T21:44:33.648493Z",
  "self": "/vectorstore/collections/summ"
}


## Publishing Data into Embeddings Collections

Now that we have created collections for embeddings, we can publish the actual embeddings into those collections.

To do this, you need a Gen3 Embeddings Manifests. Conveniently, there are examples in the Gen3 Python SDK/CLI repo in the tests folder you can use.

In [52]:
!gen3 $ai embeddings publish --help

Usage: gen3 ai embeddings publish [OPTIONS] MANIFEST_FILE

  Publish a manifest of already-embedded objects to a Gen3 collection. The
  manifest must be a CSV/TSV with the following columns:

                        embedding: JSON-encoded list of floats
  collection_id / collection_name: Target collection (use name to lookup id)
               **metadata columns: any additional columns are treated as metadata for the embedding

  Example usage:

      gen3 ai embeddings publish --collection "ctds-github-md" --manifest_file embeddings.tsv

Options:
  --batch-size INTEGER          max number of embeddings to collect before
                                pushing to API in bulk per collection
  --default-collection TEXT     Name of the default embeddings collection
                                (used when rows don't specify).
  -o, --out-manifest-file TEXT  Output filename for final manifest
  --help                        Show this message and exit.


See the above help message to understand what we need to publish embeddings. 

**tl;dr** we need a manifest file with a row per embedding. That row needs to contain the vector (embedding), along with other metadata.

**TODO: REPLACE THESE WITH TEST FILES**

In [53]:
# here's a quick visualization of the columns/data of this input manifest
df = pd.read_csv('../../tests/embeddings_tests/expr.tsv', sep='\t', nrows=3)
df

,embedding,authz,collection_name,collection_id,case_id,file_id,model
0,"[-0.2024536281824112, 0.8700736165046692, 0.63...",/programs/dev/projects/testproject1,expr,NaN,TCGA-05-4244,e0e055b6-6800-40e7-bde5-718823408f0c,expr
1,"[-0.21098750829696655, 0.8999685049057007, 0.6...",/programs/dev/projects/testproject1,expr,NaN,TCGA-05-4249,258b0b5e-2b09-4378-9606-83955ca19d7c,expr
2,"[-0.08159218728542328, 0.9297925233840942, 0.8...",/programs/dev/projects/testproject1,expr,NaN,TCGA-05-4250,f0395da6-5f12-4a35-8aa6-0b1b47a2acba,expr


In [54]:
!gen3 $ai embeddings publish ../../tests/embeddings_tests/expr.tsv --default-collection expr

Publishing Gen3 Embeddings Manifest: 100%|█| 8203/8203 [00:04<00:00, 1664.85it/s
Published 8203 embeddings.
Wrote output manifest: ../../tests/embeddings_tests/expr_output.tsv.


In [55]:
# here's a quick visualization of the columns/data of this input manifest
df = pd.read_csv('../../tests/embeddings_tests/hist.tsv', sep='\t', nrows=3)
df

,embedding,authz,collection_name,collection_id,case_id,file_id,model
0,"[-0.13619163632392883, -0.21455396711826324, 0...",/programs/dev/projects/testproject1,hist,NaN,TCGA-05-4244,TCGA-05-4244-01Z-00-DX1.d4ff32cd-38cf-40ea-821...,hist
1,"[-0.2422751635313034, -0.19156047701835632, 0....",/programs/dev/projects/testproject1,hist,NaN,TCGA-05-4249,TCGA-05-4249-01Z-00-DX1.9fce0297-cc19-4c04-872...,hist
2,"[-0.15457499027252197, 0.021765941753983498, 0...",/programs/dev/projects/testproject1,hist,NaN,TCGA-05-4250,TCGA-05-4250-01Z-00-DX1.90f67fdf-dff9-46ca-af7...,hist


In [56]:
!gen3 $ai embeddings publish ../../tests/embeddings_tests/hist.tsv --default-collection hist

Publishing Gen3 Embeddings Manifest: 100%|█| 9751/9751 [00:30<00:00, 320.94it/s]
Published 9751 embeddings.
Wrote output manifest: ../../tests/embeddings_tests/hist_output.tsv.


In [57]:
# here's a quick visualization of the columns/data of this input manifest
df = pd.read_csv('../../tests/embeddings_tests/summ.tsv', sep='\t', nrows=3)
df

,embedding,authz,collection_name,collection_id,case_id,file_id,model
0,"[0.0166015625, -0.0108642578125, 0.00028991699...",/programs/dev/projects/testproject1,summ,NaN,TCGA-02-2466,TCGA-02-2466.e9e97b51-1474-463b-8693-7b66f74319c9,summ
1,"[0.00726318359375, -0.007080078125, -0.0030670...",/programs/dev/projects/testproject1,summ,NaN,TCGA-02-2470,TCGA-02-2470.e21f66d9-e124-43d7-81fe-489d15d69cbf,summ
2,"[0.022216796875, -0.0111083984375, -0.01599121...",/programs/dev/projects/testproject1,summ,NaN,TCGA-02-2483,TCGA-02-2483.e73f6ba1-564c-4fea-b088-f2357ff49ee7,summ


In [58]:
!gen3 $ai embeddings publish ../../tests/embeddings_tests/summ.tsv --default-collection summ

Publishing Gen3 Embeddings Manifest: 100%|█| 9523/9523 [00:58<00:00, 162.21it/s]
Published 9523 embeddings.
Wrote output manifest: ../../tests/embeddings_tests/summ_output.tsv.


## Convert Published Embeddings Manifests into Indexing Manifests

Each of the `publish` commands above generated an output file `{input_filename}_output.tsv` (unless you overrode the output filename). 

Those outputs are manifests that now contain the final `embedding_id` and other Gen3 Embedding information that came back from creating embeddings through the Gen3 Embeddings API.

We can **convert** those _outputted_ Published Gen3 Embeddings Manifests into Gen3 Indexing Manifests to _input_ into the Gen3 Indexing process. This will allow us to create persistent, globally unique identifiers (GUIDs) in Gen3 through the Gen3 Indexing API.

So first, let's convert to the expected format for indexing.

In [59]:
# here's a quick visualization of the columns/data of the output manifest from previous `publish` commands
# this is what we'll convert
df = pd.read_csv('../../tests/embeddings_tests/expr_output.tsv', sep='\t', nrows=3)
df

,embedding_id,embedding,collection_id,authz,self,case_id,file_id,model
0,65249fd6-95d2-4dc5-b399-24d2dcddce31,"[-0.17901486158370972, 0.6868948340415955, 1.3...",141,['/vectorstore/collections/expr'],/vectorstore/collections/expr/embeddings/65249...,TCGA-XR-A8TG,9ea1ae11-2d1e-4d55-bbc6-c6f136041c6d,expr
1,1793545d-cc57-40ca-ab32-e26d6e010bb6,"[-0.19281341135501862, 0.7919774055480957, 1.1...",141,['/vectorstore/collections/expr'],/vectorstore/collections/expr/embeddings/17935...,TCGA-XS-A8TJ,f3491776-22a7-4fac-baf0-2d2995b42572,expr
2,73a2b78d-4bb4-4327-bcaa-ca94f527e276,"[-0.1024128794670105, 0.7253023982048035, 0.92...",141,['/vectorstore/collections/expr'],/vectorstore/collections/expr/embeddings/73a2b...,TCGA-XT-AASU,aed00751-a16b-4452-a69b-af47d910b5b5,expr


In [60]:
!gen3 $ai embeddings convert --help

Usage: gen3 ai embeddings convert [OPTIONS] MANIFEST_FILE

  Convert a Gen3 Embeddings Manifest into a Gen3 Indexing Manifest.

  The input manifest (Gen3 Embeddings Manifest) must contain at least the columns: 
 
      embedding: JSON-encoded list of floats
      authz: single Gen3 authz resource path which will control authorization to this embedding
      self: URL self-representation of the embedding in the Gen3 Embeddings API
 
  The output file (Gen3 Indexing Manifest) will have the columns:
 
      guid    md5    size    authz    acl    url
 
  The GUID is left blank (it will be generated by the indexing), `md5` and `size` are
  derived from the JSON representation of the embedding vector, `authz` is
  passed through from the input, and `url` is populated from a `self`
  column.
 
  A new file named `<original>_converted.tsv` is created in the same directory as
  the source file.
 
  Example:
      gen3 ai embeddings convert ./tests/embeddings_tests/test_summ.tsv

Options:
  -o,

In [61]:
!gen3 $ai embeddings convert ../../tests/embeddings_tests/expr_output.tsv --url-prefix $url_prefix
!gen3 $ai embeddings convert ../../tests/embeddings_tests/hist_output.tsv --url-prefix $url_prefix
!gen3 $ai embeddings convert ../../tests/embeddings_tests/summ_output.tsv --url-prefix $url_prefix

Writing converted manifest to '../../tests/embeddings_tests/expr_output_converted.tsv'
Converting rows: 100%|██████████████████████| 203/203 [00:00<00:00, 8026.88it/s]
Done! Check for errors above. Gen3 Indexing Manifest: ../../tests/embeddings_tests/expr_output_converted.tsv
Writing converted manifest to '../../tests/embeddings_tests/hist_output_converted.tsv'
Converting rows: 100%|██████████████████████| 751/751 [00:00<00:00, 1462.01it/s]
Done! Check for errors above. Gen3 Indexing Manifest: ../../tests/embeddings_tests/hist_output_converted.tsv
Writing converted manifest to '../../tests/embeddings_tests/summ_output_converted.tsv'
Converting rows: 100%|███████████████████████| 523/523 [00:00<00:00, 741.44it/s]
Done! Check for errors above. Gen3 Indexing Manifest: ../../tests/embeddings_tests/summ_output_converted.tsv


That `convert` command created new `{original_filename}_converted.tsv` files. Let's take a look at those:

In [62]:
df = pd.read_csv('../../tests/embeddings_tests/expr_output_converted.tsv', sep='\t', nrows=3)
df

,guid,md5,size,authz,acl,url
0,NaN,79e380cac79ff199cd1d9cfa9c386199,5074,['/vectorstore/collections/expr'],NaN,http://127.0.0.1:4142/vectorstore/collections/...
1,NaN,d592a94b9b3d2e2d519e9b171ac1763e,5090,['/vectorstore/collections/expr'],NaN,http://127.0.0.1:4142/vectorstore/collections/...
2,NaN,5d5fbf88f62fc3b4baabf6fa106ae71e,5087,['/vectorstore/collections/expr'],NaN,http://127.0.0.1:4142/vectorstore/collections/...


The above is a Gen3 Indexing Manifest with no GUIDs - we'll let the Gen3 Indexing command and backend generate those for us.

The `md5` checksum and `size` in bytes columns above are of the JSON-stringified version of the vector (in other words, the "data" is the vector itself).

`url` is a direct link to the Gen3 Embeddings API for that particular embedding.

Now, before we index, let's validate our Gen3 Indexing manifest is correctly formatted:

In [63]:
!gen3 objects manifest validate-manifest-format ../../tests/embeddings_tests/expr_output_converted.tsv --allowed-protocols https --allowed-protocols http
!gen3 objects manifest validate-manifest-format ../../tests/embeddings_tests/hist_output_converted.tsv --allowed-protocols https --allowed-protocols http
!gen3 objects manifest validate-manifest-format ../../tests/embeddings_tests/summ_output_converted.tsv --allowed-protocols https --allowed-protocols http

In [64]:
!gen3 objects manifest publish --help

Usage: gen3 objects manifest publish [OPTIONS] [FILE]

  Publishes specified object manifest to Gen3 instance.

Options:
  --thread-num INTEGER            number of threads for indexing  [default: 8]
  --append-urls                   If supplied, will append urls for existing
                                  records. e.g. existing urls will still exist
                                  and new ones will be added
                                  
                                  By default the newly provided urls will
                                  REPLACE existing urls
  --manifest-file-delimiter TEXT  string character that delimites the file
                                  (tab or comma). Defaults to tab.
  --out-manifest-file TEXT        The path to output manifest  [default:
                                  indexing-output-manifest.csv]
  --force-metadata-columns-even-if-empty
                                  force the creation of a metadata column
                        

TODO: fix permissions and continue writing the notebook to

- publish indexing manifest
- get list of GUIDs from manifest
- simulate a search resulting in GUIDs
- bulk retrieve embeddings from Fence


In [66]:
!gen3 objects manifest publish ../../tests/embeddings_tests/expr_output_converted.tsv --out-manifest-file ../../tests/embeddings_tests/expr_output_converted_indexed.tsv

Publishing/writing object data from ../../tests/embeddings_tests/expr_output_converted.tsv...
    to: https://avantol-local.dev.planx-pla.net
Traceback (most recent call last):
  File "/Users/avantol/Documents/repos/gen3sdk-python/gen3/tools/indexing/index_manifest.py", line 291, in _index_record
    doc = indexclient.create(**record)
  File "/Users/avantol/Library/Caches/pypoetry/virtualenvs/gen3-NXImaaEr-py3.13/lib/python3.13/site-packages/indexclient/client.py", line 291, in create
    resp = self._post(
        "index/",
    ...<2 lines>...
        auth=self.auth,
    )
  File "/Users/avantol/Library/Caches/pypoetry/virtualenvs/gen3-NXImaaEr-py3.13/lib/python3.13/site-packages/indexclient/client.py", line 43, in timeout
    return func(*args, **kwargs)
  File "/Users/avantol/Library/Caches/pypoetry/virtualenvs/gen3-NXImaaEr-py3.13/lib/python3.13/site-packages/indexclient/client.py", line 417, in _post
    handle_error(resp)
    ~~~~~~~~~~~~^^^^^^
  File "/Users/avantol/Library/Cach